# Notebook 10 — Battery Storage

Battery energy storage systems (BESS) provide flexibility by shifting energy between cheap and expensive hours. This notebook explores the storage model.

Topics:
1. `StorageUnit` — physical model (capacity, efficiency, SOC, self-discharge)
2. Rolling-percentile arbitrage — the dispatch strategy
3. SOC evolution over a year
4. Revenue vs capacity sizing
5. Impact on electricity prices

**Runtime**: ~60 seconds

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from energy_sim.config import (
    ITALIAN_MIX, GAS_SCENARIOS, P_PEAK_GW,
    QUARTERS_PER_DAY, STORAGE_UNITS,
    STORAGE_PERCENTILE_WINDOW_QH,
    STORAGE_CHARGE_PERCENTILE, STORAGE_DISCHARGE_PERCENTILE,
)
from energy_sim.storage import StorageUnit, build_storage_units
from energy_sim.simulation import run_monte_carlo, sweep_storage_capacity

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

## 1. The StorageUnit

The default BESS is a 2 GW / 4 GWh aggregated unit (consistent with Italy's PNIEC 2030 targets):

In [ ]:
units = build_storage_units(STORAGE_UNITS)
su = units[0]

print(f"Storage unit: {su.name}")
print(f"  Power capacity:     {su.power_capacity_gw:.1f} GW")
print(f"  Energy capacity:    {su.energy_capacity_gwh:.1f} GWh")
print(f"  Duration:           {su.energy_capacity_gwh / su.power_capacity_gw:.1f} hours")
print(f"  Round-trip eff:     {su.efficiency_roundtrip:.0%}")
print(f"  Per-leg eff:        {su.eta_charge:.3f} (charge) / {su.eta_discharge:.3f} (discharge)")
print(f"  SOC band:           [{su.soc_min_frac:.0%}, {su.soc_max_frac:.0%}]")
print(f"  Usable capacity:    {su.energy_capacity_gwh * (su.soc_max_frac - su.soc_min_frac):.1f} GWh")
print(f"  Self-discharge:     {su.self_discharge_per_day:.1%} per day")
print(f"  Synthetic inertia:  {su.h_synthetic:.1f} s")

print(f"\nArbitrage parameters:")
print(f"  Window:   {STORAGE_PERCENTILE_WINDOW_QH} qh = {STORAGE_PERCENTILE_WINDOW_QH/96:.0f} days")
print(f"  Charge:   price < {STORAGE_CHARGE_PERCENTILE}th percentile")
print(f"  Discharge: price > {STORAGE_DISCHARGE_PERCENTILE}th percentile")

## 2. Dispatch with storage

Let's run a simulation with and without storage and compare:

In [ ]:
# Without storage
mc_no_stor = run_monte_carlo(
    ITALIAN_MIX, GAS_SCENARIOS['base'],
    n_runs=10, seed=42,
)

# With storage
mc_stor = run_monte_carlo(
    ITALIAN_MIX, GAS_SCENARIOS['base'],
    n_runs=10, seed=42,
    storage_cfg=STORAGE_UNITS,
)

print(f"{'Metric':<30s} {'No storage':>12s} {'With storage':>12s} {'Delta':>10s}")
print("-" * 70)
for metric, key, fmt, scale in [
    ('Avg price (EUR/MWh)', 'avg_price', '.1f', 1),
    ('Carbon intensity (gCO2/kWh)', 'carbon_intensity', '.0f', 1),
    ('Curtailment (p.u.-qh)', 'curtailment', '.1f', 1),
]:
    v_no = mc_no_stor[key].mean() / scale
    v_st = mc_stor[key].mean() / scale
    print(f"{metric:<30s} {v_no:>12{fmt}} {v_st:>12{fmt}} {v_st-v_no:>+10{fmt}}")

# Storage-specific metrics
rev = mc_stor['storage_revenue_eur'][:, 0]
cycles = mc_stor['storage_equivalent_cycles'][:, 0]
avg_soc = mc_stor['storage_avg_soc'][:, 0]

print(f"\nStorage metrics (mean +/- std over MC runs):")
print(f"  Revenue:    {rev.mean()/1e6:+.1f} +/- {rev.std()/1e6:.1f} M EUR/year")
print(f"  Eq. cycles: {cycles.mean():.0f} +/- {cycles.std():.0f} per year")
print(f"  Avg SOC:    {avg_soc.mean():.2f} +/- {avg_soc.std():.3f}")

## 3. SOC evolution

Let's look at the state-of-charge pattern over the year and the charge/discharge behavior:

In [ ]:
# Monthly SOC pattern
monthly_soc = mc_stor['storage_monthly_avg_soc'].mean(axis=0)[0]  # first unit

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(month_names, monthly_soc, color='teal', alpha=0.7)
ax.axhline(su.soc_min_frac, color='red', ls='--', alpha=0.5, label=f'SOC min ({su.soc_min_frac:.0%})')
ax.axhline(su.soc_max_frac, color='green', ls='--', alpha=0.5, label=f'SOC max ({su.soc_max_frac:.0%})')
ax.set_ylabel('Average SOC')
ax.set_title('Monthly average state of charge')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Revenue vs capacity sizing

The key economic question: what is the **optimal storage duration** (GWh per GW of power)?

In [ ]:
cap_sweep = sweep_storage_capacity(
    ITALIAN_MIX, GAS_SCENARIOS['base'],
    power_gw=2.0,
    energy_gwh_range=np.array([1, 2, 4, 8, 12]),
    n_runs=10, seed=42,
)

In [ ]:
gwh_vals = [r['energy_gwh'] for r in cap_sweep]
rev_vals = [r['revenue_mean'] / 1e6 for r in cap_sweep]
rev_stds = [r['revenue_std'] / 1e6 for r in cap_sweep]
price_vals = [r['price_mean'] for r in cap_sweep]
cycle_vals = [r['equiv_cycles'] for r in cap_sweep]
soc_vals = [r['avg_soc'] for r in cap_sweep]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Revenue
axes[0, 0].errorbar(gwh_vals, rev_vals, yerr=rev_stds, fmt='o-',
                     color='green', capsize=4, lw=2)
axes[0, 0].set_xlabel('Energy capacity (GWh)')
axes[0, 0].set_ylabel('Revenue (M EUR/year)')
axes[0, 0].set_title('Annual revenue vs energy capacity (2 GW power)')

# Revenue per GWh (diminishing returns)
rev_per_gwh = [r / g for r, g in zip(rev_vals, gwh_vals)]
axes[0, 1].plot(gwh_vals, rev_per_gwh, 'o-', color='orange', lw=2)
axes[0, 1].set_xlabel('Energy capacity (GWh)')
axes[0, 1].set_ylabel('Revenue per GWh (M EUR/GWh/year)')
axes[0, 1].set_title('Marginal value of storage duration')

# Equivalent cycles
axes[1, 0].plot(gwh_vals, cycle_vals, 'o-', color='teal', lw=2)
axes[1, 0].set_xlabel('Energy capacity (GWh)')
axes[1, 0].set_ylabel('Equivalent full cycles/year')
axes[1, 0].set_title('Cycle count vs capacity')

# Electricity price impact
axes[1, 1].plot(gwh_vals, price_vals, 'o-', color='steelblue', lw=2)
axes[1, 1].set_xlabel('Energy capacity (GWh)')
axes[1, 1].set_ylabel('Avg electricity price (EUR/MWh)')
axes[1, 1].set_title('Price impact of storage')

plt.tight_layout()
plt.show()

## 5. Play with parameters

In [ ]:
# ── PLAY WITH THESE ──────────────────────────────────────
custom_power_gw = 4.0         # try 1-8 GW
custom_energy_gwh = 8.0       # try 2-16 GWh
custom_efficiency = 0.92      # try 0.80-0.95
# ─────────────────────────────────────────────────────────

custom_cfg = {
    'custom_bess': {
        'energy_capacity_gwh': custom_energy_gwh,
        'power_capacity_gw': custom_power_gw,
        'efficiency_roundtrip': custom_efficiency,
    }
}

mc_custom = run_monte_carlo(
    ITALIAN_MIX, GAS_SCENARIOS['base'],
    n_runs=10, seed=42,
    storage_cfg=custom_cfg,
)

rev_c = mc_custom['storage_revenue_eur'][:, 0]
cyc_c = mc_custom['storage_equivalent_cycles'][:, 0]

print(f"Custom BESS ({custom_power_gw} GW / {custom_energy_gwh} GWh, eta={custom_efficiency:.0%}):")
print(f"  Avg price:  {mc_custom['avg_price'].mean():.1f} EUR/MWh")
print(f"  Revenue:    {rev_c.mean()/1e6:+.1f} M EUR/year")
print(f"  Cycles:     {cyc_c.mean():.0f}/year")
print(f"  Duration:   {custom_energy_gwh/custom_power_gw:.1f} hours")

## Key Takeaways

1. **Rolling-percentile arbitrage** is a simple but effective strategy: charge when prices are in the bottom quartile, discharge when in the top quartile. No lookahead needed.
2. **Revenue has diminishing returns** with energy capacity — the first GWh is the most valuable. Beyond ~4h duration, the battery sits idle too often.
3. **Storage reduces price volatility** — it compresses the spread between cheap and expensive hours, slightly lowering average prices.
4. **Cycle count** decreases with larger batteries (fewer full charge/discharge events) — this is good for battery lifetime but bad for per-GWh revenue.
5. **Round-trip efficiency** (88% default) means ~12% of stored energy is lost. This is the physical cost of time-shifting.
6. **Synthetic inertia** (H=4s) lets the BESS contribute to grid stability — valuable at high renewable penetration.

**Next notebook**: [11 — Full Analysis Pipeline](./11_full_analysis_pipeline.ipynb) — putting everything together.